# Business entity resolution — hybrid retrieval and matching

Run top to bottom. This notebook builds a disk-backed SQLite FTS5 index for Source 2 and Source 3, adds multilingual embedding retrieval with a compressed FAISS index, trains a LightGBM pair classifier, measures macro F0.5, and writes both required test TSV files.

The full data has millions of target rows. The embedding encoder runs on GPU, while FTS, FAISS search, and LightGBM use CPU. Indexes are checkpointed in CACHE_DIR. A hosted runtime's local disk may disappear when the session ends, so use persistent storage for the cache if you need to resume across sessions. No external business lookup is used.


## 1. Check and install runtime packages

In [ ]:
import importlib.util
import subprocess
import sys

ENABLE_EMBEDDINGS = True
required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "rapidfuzz": "rapidfuzz",
    "lightgbm": "lightgbm",
    "sklearn": "scikit-learn",
}
if ENABLE_EMBEDDINGS:
    required.update({
        "torch": "torch",
        "sentence_transformers": "sentence-transformers",
        "faiss": "faiss-cpu",
    })
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are installed.")


In [ ]:
import csv
import gc
import hashlib
import os
import re
import shutil
import sqlite3
import unicodedata
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sklearn.model_selection import train_test_split

if ENABLE_EMBEDDINGS:
    import faiss
    import torch
    from sentence_transformers import SentenceTransformer

print("Python:", sys.version.split()[0])
print("SQLite:", sqlite3.sqlite_version)
with sqlite3.connect(":memory:") as probe:
    probe.execute("CREATE VIRTUAL TABLE fts_probe USING fts5(name, address)")
print("SQLite FTS5: available")
if ENABLE_EMBEDDINGS:
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())


## 2. Paths and controls

The original notebook path is included in the search. On a hosted runtime, upload or mount the challenge data, then set DATASET_DIR to its dataset folder. Set CACHE_DIR_OVERRIDE to a persistent folder if you need checkpoints across sessions; local runtime storage is usually faster. TRAIN_S1_SAMPLE and VALID_S1_SAMPLE control model development cost; all target records remain indexed, and test inference covers every Source 1 row.


In [ ]:
DATASET_DIR = None  # Example: Path("/kaggle/input/your-dataset/dataset")
CACHE_DIR_OVERRIDE = None  # Optional persistent path for resumable indexes
VERIFY_FULL_DATASET = True  # Set False only when intentionally testing a subset
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_MODEL_REVISION = "383bbb10e05bb8885acd71a7956dc2d823b60305"
EMBEDDING_BATCH_SIZE = 256
EMBEDDING_INDEX_BATCH_ROWS = 20_000
EMBEDDING_TRAIN_ROWS = 120_000
EMBEDDING_NLIST = 4_096
EMBEDDING_PQ_M = 48
EMBEDDING_NPROBE = 24
EMBEDDING_TOP_K = 20
EMBEDDING_CHECKPOINT_ROWS = 500_000
RUN_TEST_INFERENCE = True
TRAIN_S1_SAMPLE = 30_000
VALID_S1_SAMPLE = 6_000
NEGATIVES_PER_TRAIN_S1 = 20
TOP_K_PER_CHANNEL = 20
MAX_FINAL_CANDIDATES = 50
INDEX_CHUNK_SIZE = 100_000
INFERENCE_BATCH_SIZE = 256
RANDOM_SEED = 42

if DATASET_DIR is None:
    search_roots = [
        Path("../../student_resource/dataset"),
        Path("student_resource/dataset"),
        Path("dataset"),
        Path("/kaggle/input"),
        Path("/content"),
    ]
    direct = [p for p in search_roots if (p / "train/train_source1.tsv").is_file()]
    if not direct:
        direct = [
            p.parent.parent for root in search_roots[-2:] if root.exists()
            for p in root.glob("**/train/train_source1.tsv")
            if (p.parent.parent / "test/test_source1.tsv").is_file()
        ]
    if not direct:
        raise FileNotFoundError("Set DATASET_DIR to the folder containing train/ and test/.")
    DATASET_DIR = direct[0]
DATASET_DIR = Path(DATASET_DIR).resolve()
CACHE_DIR = Path(CACHE_DIR_OVERRIDE) if CACHE_DIR_OVERRIDE else Path.cwd() / "er_cache"
OUTPUT_DIR = Path.cwd() / "output"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Dataset:", DATASET_DIR)
print("Cache:", CACHE_DIR)
print("Free disk (GB):", round(shutil.disk_usage(CACHE_DIR).free / 1e9, 1))


## 3. Check the copied dataset

The complete challenge files have known byte sizes and row counts. This quick check catches a partial upload or the wrong dataset folder before a long index build. Set VERIFY_FULL_DATASET=False only when deliberately testing with smaller files.


In [ ]:
import codecs

EXPECTED_FILES = {
    "train/train_source1.tsv": (210_069_713, 2_206_821),
    "train/train_source2.tsv": (489_301_488, 5_034_616),
    "train/train_source3.tsv": (503_705_637, 5_285_603),
    "train/train_ground_truth.tsv": (127_015_583, 2_206_821),
    "test/test_source1.tsv": (175_022_086, 1_732_544),
    "test/test_source2.tsv": (509_456_422, 4_887_273),
    "test/test_source3.tsv": (506_002_772, 5_082_316),
}

def check_dataset_files(dataset_dir):
    problems = []
    for relative, (expected_bytes, _) in EXPECTED_FILES.items():
        path = dataset_dir / relative
        if not path.is_file():
            problems.append(f"{path}: file is missing")
        elif path.stat().st_size != expected_bytes:
            problems.append(
                f"{path}: {path.stat().st_size:,} bytes; "
                f"expected {expected_bytes:,} bytes"
            )
    if problems:
        raise ValueError(
            "The mounted dataset differs from the complete challenge files. "
            "Check DATASET_DIR and copy the original TSV files again:\n"
            + "\n".join(problems)
        )

    for relative, (_, expected_rows) in EXPECTED_FILES.items():
        path = dataset_dir / relative
        decoder = codecs.getincrementaldecoder("utf-8")()
        newline_count = 0
        bytes_read = 0
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                try:
                    decoder.decode(chunk)
                except UnicodeDecodeError as error:
                    raise ValueError(
                        f"{path} has invalid UTF-8 near byte {bytes_read + error.start:,}. "
                        "Copy the original file again."
                    ) from error
                newline_count += chunk.count(b"\n")
                bytes_read += len(chunk)
        try:
            decoder.decode(b"", final=True)
        except UnicodeDecodeError as error:
            raise ValueError(
                f"{path} ends with an incomplete UTF-8 character. "
                "Copy the original file again."
            ) from error
        actual_rows = newline_count - 1
        if actual_rows != expected_rows:
            raise ValueError(
                f"{path} has {actual_rows:,} rows; expected {expected_rows:,}. "
                "Copy the original file again."
            )
        print(f"Verified {relative}: {actual_rows:,} rows")

if VERIFY_FULL_DATASET:
    check_dataset_files(DATASET_DIR)


## 4. Normalize fields

Only challenge data is used. Raw fields are retained in the original TSVs; the index stores normalized forms for retrieval and matching.


In [ ]:
QUERY_STOPWORDS = {
    "the", "and", "for", "with", "company", "corporation", "corp",
    "private", "pvt", "limited", "ltd", "inc", "llp", "llc",
    "road", "street", "avenue", "near", "building", "floor",
}
TOKEN_RE = re.compile(r"[^\W_]+", re.UNICODE)
NUMBER_RE = re.compile(r"\d+", re.UNICODE)

def normalize_text(value):
    value = unicodedata.normalize("NFKD", str(value or ""))
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = value.casefold().replace("&", " and ")
    return " ".join(TOKEN_RE.findall(value))

def prepare_record(row):
    return (
        row.entity_id,
        normalize_text(row.business_name),
        normalize_text(row.business_address),
        normalize_text(row.country),
    )

def informative_tokens(text, limit=3):
    tokens = list(dict.fromkeys(TOKEN_RE.findall(text)))
    useful = [t for t in tokens if len(t) >= 3 and t not in QUERY_STOPWORDS]
    if not useful:
        useful = [t for t in tokens if len(t) >= 2]
    return sorted(useful, key=lambda t: (-len(t), t))[:limit]

print(normalize_text("Orelee’s Barber-Shop & Co."))


## 5. Multilingual encoder

The selected MiniLM model is Apache 2.0 licensed and produces 384-dimensional embeddings. Texts use only the supplied name and address fields. The first run downloads model weights; later runs can reuse the local model cache.


In [ ]:
if ENABLE_EMBEDDINGS:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "ENABLE_EMBEDDINGS=True requires a CUDA GPU for the full dataset. "
            "Enable a GPU runtime or set ENABLE_EMBEDDINGS=False for a CPU-only baseline."
        )
    embedding_model = SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        revision=EMBEDDING_MODEL_REVISION,
        device="cuda",
    )
    EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
    print("Encoder:", EMBEDDING_MODEL_NAME, "dimension:", EMBEDDING_DIM)
else:
    embedding_model = None
    EMBEDDING_DIM = 0

def embedding_text(name, address):
    return f"name: {name} address: {address}"

def encode_texts(texts):
    vectors = embedding_model.encode(
        texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return np.ascontiguousarray(vectors, dtype=np.float32)

def encode_source_frame(frame):
    if not ENABLE_EMBEDDINGS:
        return None
    texts = [
        embedding_text(normalize_text(row.business_name), normalize_text(row.business_address))
        for row in frame.itertuples(index=False)
    ]
    return encode_texts(texts)


## 6. Build reusable target indexes

One FTS5 database is built for training S2/S3 and another for test S2/S3. The database signature prevents accidental reuse with different files. The build reads TSVs in chunks, so it does not load all target rows into memory.


In [ ]:
SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]

def source_paths(split):
    return [
        DATASET_DIR / split / f"{split}_source2.tsv",
        DATASET_DIR / split / f"{split}_source3.tsv",
    ]

def index_signature(paths):
    return "|".join(
        f"{p.resolve()}:{p.stat().st_size}:{p.stat().st_mtime_ns}" for p in paths
    )

def build_index(split):
    paths = source_paths(split)
    signature = index_signature(paths)
    db_path = CACHE_DIR / f"{split}_targets.sqlite"
    if db_path.exists():
        with sqlite3.connect(db_path) as check:
            try:
                row = check.execute("SELECT value FROM metadata WHERE key='signature'").fetchone()
                if row and row[0] == signature:
                    print(f"Reusing {db_path}")
                    return db_path
            except sqlite3.DatabaseError:
                pass
        print(f"Rebuilding stale or incomplete cache: {db_path}")
        db_path.unlink()

    print(f"Building {split} FTS index at {db_path}")
    con = sqlite3.connect(db_path)
    con.execute("PRAGMA journal_mode=OFF")
    con.execute("PRAGMA synchronous=OFF")
    con.execute("PRAGMA temp_store=MEMORY")
    con.execute("PRAGMA cache_size=-65536")
    con.execute("CREATE TABLE metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)")
    con.execute(
        "CREATE VIRTUAL TABLE records USING fts5("
        "entity_id UNINDEXED, name, address, country UNINDEXED, "
        "detail=column, tokenize='unicode61 remove_diacritics 2')"
    )
    total = 0
    try:
        for path in paths:
            print("Indexing", path.name)
            reader = pd.read_csv(
                path, sep="\t", dtype=str, na_filter=False,
                usecols=SOURCE_COLUMNS, chunksize=INDEX_CHUNK_SIZE,
            )
            for chunk in reader:
                rows = (prepare_record(row) for row in chunk.itertuples(index=False))
                con.executemany(
                    "INSERT INTO records(entity_id,name,address,country) VALUES (?,?,?,?)",
                    rows,
                )
                con.commit()
                total += len(chunk)
                if total % 500_000 < len(chunk):
                    print(f"  {total:,} target rows indexed")
        con.execute("INSERT INTO metadata(key,value) VALUES('signature',?)", (signature,))
        con.commit()
    finally:
        con.close()
    print(f"{split}: {total:,} target rows indexed")
    return db_path

train_db_path = build_index("train")


## 7. Build compressed embedding indexes

FAISS stores compact product-quantized vectors keyed by the SQLite row ID. It retrieves semantic candidates and reconstructs approximate vectors for the classifier's cosine feature. Each checkpoint is reusable after a runtime interruption. Full target encoding is the longest stage of this notebook.


In [ ]:
def build_embedding_index(db_path, split):
    if not ENABLE_EMBEDDINGS:
        return None
    faiss.omp_set_num_threads(min(os.cpu_count() or 4, 8))
    con = sqlite3.connect(db_path)
    target_count = con.execute("SELECT count(*) FROM records").fetchone()[0]
    source_signature = con.execute(
        "SELECT value FROM metadata WHERE key='signature'"
    ).fetchone()[0]
    settings = (
        f"{source_signature}|{EMBEDDING_MODEL_NAME}|{EMBEDDING_MODEL_REVISION}|"
        f"{EMBEDDING_NLIST}|{EMBEDDING_PQ_M}"
    )
    digest = hashlib.sha256(settings.encode("utf-8")).hexdigest()[:16]
    index_path = CACHE_DIR / f"{split}_embeddings_{digest}.faiss"

    if index_path.exists():
        index = faiss.read_index(str(index_path))
        if index.d != EMBEDDING_DIM or index.ntotal > target_count:
            raise ValueError("Saved FAISS index is incompatible with the current data/model")
        print(f"Resuming {split} FAISS index at {index.ntotal:,}/{target_count:,}")
    else:
        sample_per_half = min(EMBEDDING_TRAIN_ROWS // 2, max(1, target_count // 2))
        first = con.execute(
            "SELECT name,address FROM records WHERE rowid<=? LIMIT ?",
            (target_count // 2, sample_per_half),
        ).fetchall()
        second = con.execute(
            "SELECT name,address FROM records WHERE rowid>? LIMIT ?",
            (target_count // 2, sample_per_half),
        ).fetchall()
        sample_texts = [embedding_text(name, address) for name, address in first + second]
        if not sample_texts:
            raise ValueError("No target records available for the embedding index")
        train_vectors = encode_texts(sample_texts)
        if len(train_vectors) < 10_000:
            index = faiss.IndexIDMap2(faiss.IndexFlatIP(EMBEDDING_DIM))
        else:
            if EMBEDDING_DIM % EMBEDDING_PQ_M:
                raise ValueError("EMBEDDING_PQ_M must divide the model embedding dimension")
            nlist = min(EMBEDDING_NLIST, max(1, len(train_vectors) // 40))
            index = faiss.IndexIVFPQ(
                faiss.IndexFlatIP(EMBEDDING_DIM),
                EMBEDDING_DIM, nlist, EMBEDDING_PQ_M, 8,
                faiss.METRIC_INNER_PRODUCT,
            )
            print(f"Training FAISS IVF-PQ: {nlist} lists, {len(train_vectors):,} vectors")
            index.train(train_vectors)
            index.set_direct_map_type(faiss.DirectMap.Hashtable)
            index.nprobe = min(EMBEDDING_NPROBE, nlist)
        del train_vectors, sample_texts, first, second
        gc.collect()
        initial_checkpoint = index_path.with_suffix(".tmp.faiss")
        faiss.write_index(index, str(initial_checkpoint))
        os.replace(initial_checkpoint, index_path)

    if index.ntotal == target_count:
        con.close()
        print(f"{split} embedding index ready: {index.ntotal:,} vectors")
        return index

    if hasattr(index, "nprobe"):
        index.nprobe = min(EMBEDDING_NPROBE, index.nlist)
    next_checkpoint = (
        (index.ntotal // EMBEDDING_CHECKPOINT_ROWS) + 1
    ) * EMBEDDING_CHECKPOINT_ROWS
    cursor = con.execute(
        "SELECT rowid,name,address FROM records WHERE rowid>? ORDER BY rowid",
        (int(index.ntotal),),
    )
    try:
        while True:
            batch = cursor.fetchmany(EMBEDDING_INDEX_BATCH_ROWS)
            if not batch:
                break
            ids = np.array([row[0] for row in batch], dtype=np.int64)
            vectors = encode_texts([embedding_text(row[1], row[2]) for row in batch])
            index.add_with_ids(vectors, ids)
            if index.ntotal >= next_checkpoint:
                checkpoint = index_path.with_suffix(".tmp.faiss")
                faiss.write_index(index, str(checkpoint))
                os.replace(checkpoint, index_path)
                print(f"  {split} embeddings: {index.ntotal:,}/{target_count:,}")
                next_checkpoint += EMBEDDING_CHECKPOINT_ROWS
        checkpoint = index_path.with_suffix(".tmp.faiss")
        faiss.write_index(index, str(checkpoint))
        os.replace(checkpoint, index_path)
    finally:
        con.close()
    if index.ntotal != target_count:
        raise RuntimeError(f"FAISS index has {index.ntotal:,} of {target_count:,} targets")
    print(f"{split} embedding index ready: {index.ntotal:,} vectors")
    return index

train_embedding_index = build_embedding_index(train_db_path, "train")


## 8. Candidate retrieval

The final candidate set is the union of four channels: name tokens, address tokens, a distinctive name prefix, and multilingual embedding ANN. Each channel returns a fixed top K. The capped union is exactly what the classifier scores and what candidate_pairs.tsv records. FAISS scores are approximate; the cosine feature reconstructs compressed vectors.


In [ ]:
class CandidateRetriever:
    def __init__(self, db_path, top_k=20, max_candidates=50, embedding_index=None):
        self.con = sqlite3.connect(db_path)
        self.top_k = top_k
        self.max_candidates = max_candidates
        self.embedding_index = embedding_index

    def close(self):
        self.con.close()

    def _search(self, expression, weights):
        sql = (
            "SELECT rowid,entity_id,name,address,country,bm25(records,"
            + weights + ") AS rank_score "
            "FROM records WHERE records MATCH ? "
            "ORDER BY rank_score LIMIT ?"
        )
        return self.con.execute(sql, (expression, self.top_k)).fetchall()

    @staticmethod
    def _add(candidates, row, channel, rank, score):
        rowid, target_id, target_name, target_addr, target_country = row
        entry = candidates.setdefault(target_id, {
            "rowid": int(rowid),
            "target_id": target_id,
            "target_name": target_name,
            "target_addr": target_addr,
            "target_country": target_country,
            "name_rank": 99, "address_rank": 99, "prefix_rank": 99,
            "embedding_rank": 99,
            "name_bm25": 0.0, "address_bm25": 0.0, "prefix_bm25": 0.0,
            "embedding_ann_score": 0.0, "embedding_cosine": 0.0,
        })
        if channel == "embedding":
            entry["embedding_rank"] = rank
            entry["embedding_ann_score"] = float(score)
        else:
            entry[f"{channel}_rank"] = rank
            entry[f"{channel}_bm25"] = -float(score)

    def embedding_search(self, query_vectors):
        if self.embedding_index is None or query_vectors is None:
            return None, None
        vectors = np.ascontiguousarray(query_vectors, dtype=np.float32)
        return self.embedding_index.search(vectors, EMBEDDING_TOP_K)

    def retrieve(self, prepared_source, query_vector=None, semantic_hits=None):
        _, name, address, _ = prepared_source
        channels = []
        name_terms = informative_tokens(name)
        address_terms = informative_tokens(address)
        if name_terms:
            expression = "name:(" + " OR ".join(f'"{t}"' for t in name_terms) + ")"
            channels.append(("name", expression, "0,3,0.5,0"))
            prefix = name_terms[0][:5]
            if len(prefix) >= 4 and prefix.isalnum():
                channels.append(("prefix", f"name:{prefix}*", "0,3,0.5,0"))
        if address_terms:
            expression = "address:(" + " OR ".join(f'"{t}"' for t in address_terms) + ")"
            channels.append(("address", expression, "0,0.5,3,0"))

        candidates = {}
        for channel, expression, weights in channels:
            for rank, result in enumerate(self._search(expression, weights), start=1):
                self._add(candidates, result[:5], channel, rank, result[5])

        if self.embedding_index is not None and query_vector is not None:
            vector = np.ascontiguousarray(query_vector.reshape(1, -1), dtype=np.float32)
            if semantic_hits is None:
                similarities, rowids = self.embedding_index.search(
                    vector, EMBEDDING_TOP_K
                )
                scores_for_query, ids_for_query = similarities[0], rowids[0]
            else:
                scores_for_query, ids_for_query = semantic_hits
            ranked = [(rank, int(rowid), float(score))
                      for rank, (rowid, score) in enumerate(
                          zip(ids_for_query, scores_for_query), start=1
                      ) if rowid > 0]
            if ranked:
                keys = [rowid for _, rowid, _ in ranked]
                placeholders = ",".join("?" for _ in keys)
                found = self.con.execute(
                    f"SELECT rowid,entity_id,name,address,country FROM records "
                    f"WHERE rowid IN ({placeholders})", keys
                ).fetchall()
                lookup = {int(row[0]): row for row in found}
                for rank, rowid, score in ranked:
                    if rowid in lookup:
                        self._add(candidates, lookup[rowid], "embedding", rank, score)

        ordered = sorted(
            candidates.values(),
            key=lambda c: (
                min(c["name_rank"], c["address_rank"],
                    c["prefix_rank"], c["embedding_rank"]),
                -max(c["name_bm25"], c["address_bm25"],
                     c["prefix_bm25"], c["embedding_ann_score"]),
                c["target_id"],
            ),
        )[:self.max_candidates]

        if self.embedding_index is not None and query_vector is not None and ordered:
            keys = np.array([c["rowid"] for c in ordered], dtype=np.int64)
            reconstructed = np.asarray(
                self.embedding_index.reconstruct_batch(keys), dtype=np.float32
            )
            norms = np.maximum(np.linalg.norm(reconstructed, axis=1), 1e-8)
            cosine = (reconstructed @ query_vector) / norms
            for candidate, score in zip(ordered, cosine):
                candidate["embedding_cosine"] = float(score)
        return ordered

train_retriever = CandidateRetriever(
    train_db_path, TOP_K_PER_CHANNEL, MAX_FINAL_CANDIDATES,
    train_embedding_index,
)


## 9. Pair features and labels

Fuzzy and semantic similarities are calculated only on retrieved pairs. Empty fields score zero. Country equality is a generic string comparison. Retrieval ranks and scores let LightGBM learn which channels supplied each pair.


In [ ]:
def fuzzy_score(a, b, fn):
    return float(fn(a, b)) / 100.0 if a and b else 0.0

def jaro_score(a, b):
    return float(JaroWinkler.normalized_similarity(a, b)) if a and b else 0.0

def jaccard(a, b):
    left, right = set(a.split()), set(b.split())
    return len(left & right) / len(left | right) if left and right else 0.0

def shared_numbers(a, b):
    left, right = set(NUMBER_RE.findall(a)), set(NUMBER_RE.findall(b))
    return len(left & right) / len(left | right) if left and right else 0.0

def pair_features(prepared_source, candidate):
    _, name1, addr1, country1 = prepared_source
    name2 = candidate["target_name"]
    addr2 = candidate["target_addr"]
    country2 = candidate["target_country"]
    return {
        "name_ratio": fuzzy_score(name1, name2, fuzz.ratio),
        "name_token_sort": fuzzy_score(name1, name2, fuzz.token_sort_ratio),
        "name_token_set": fuzzy_score(name1, name2, fuzz.token_set_ratio),
        "name_jaro": jaro_score(name1, name2),
        "name_jaccard": jaccard(name1, name2),
        "address_ratio": fuzzy_score(addr1, addr2, fuzz.ratio),
        "address_token_sort": fuzzy_score(addr1, addr2, fuzz.token_sort_ratio),
        "address_token_set": fuzzy_score(addr1, addr2, fuzz.token_set_ratio),
        "address_jaro": jaro_score(addr1, addr2),
        "address_jaccard": jaccard(addr1, addr2),
        "number_overlap": shared_numbers(addr1, addr2),
        "country_equal": int(bool(country1 and country2 and country1 == country2)),
        "name_exact": int(bool(name1 and name1 == name2)),
        "address_exact": int(bool(addr1 and addr1 == addr2)),
        "name_missing": int(not name1 or not name2),
        "address_missing": int(not addr1 or not addr2),
        "name_length_ratio": min(len(name1), len(name2)) / max(len(name1), len(name2), 1),
        "address_length_ratio": min(len(addr1), len(addr2)) / max(len(addr1), len(addr2), 1),
        "name_rank": candidate["name_rank"],
        "address_rank": candidate["address_rank"],
        "prefix_rank": candidate["prefix_rank"],
        "embedding_rank": candidate.get("embedding_rank", 99),
        "name_bm25": candidate["name_bm25"],
        "address_bm25": candidate["address_bm25"],
        "prefix_bm25": candidate["prefix_bm25"],
        "embedding_ann_score": candidate.get("embedding_ann_score", 0.0),
        "embedding_cosine": candidate.get("embedding_cosine", 0.0),
    }

FEATURE_COLUMNS = list(pair_features(
    prepare_record(type("Example", (), {
        "entity_id": "S1-1", "business_name": "Example",
        "business_address": "1 Main St", "country": "US"
    })()),
    {
        "target_name": "example", "target_addr": "1 main st", "target_country": "us",
        "name_rank": 1, "address_rank": 1, "prefix_rank": 1,
        "name_bm25": 1.0, "address_bm25": 1.0, "prefix_bm25": 1.0,
    }
))
print(len(FEATURE_COLUMNS), "features")


## 10. Sample Source 1 records and read their ground truth

The retrieval index contains every training S2/S3 row. Sampling only the training S1 queries makes V1 tuning practical. Increase the sample sizes after a first complete run.


In [ ]:
train_s1_path = DATASET_DIR / "train/train_source1.tsv"
truth_path = DATASET_DIR / "train/train_ground_truth.tsv"

all_s1 = pd.read_csv(
    train_s1_path, sep="\t", dtype=str, na_filter=False, usecols=SOURCE_COLUMNS
)
n_selected = min(len(all_s1), TRAIN_S1_SAMPLE + VALID_S1_SAMPLE)
selected_s1 = all_s1.sample(n=n_selected, random_state=RANDOM_SEED).reset_index(drop=True)
del all_s1
gc.collect()
train_s1, valid_s1 = train_test_split(
    selected_s1, test_size=min(VALID_S1_SAMPLE, n_selected - 1),
    random_state=RANDOM_SEED, shuffle=True,
)
train_s1 = train_s1.reset_index(drop=True)
valid_s1 = valid_s1.reset_index(drop=True)
wanted_ids = set(selected_s1.entity_id)
truth = {}
for chunk in pd.read_csv(
    truth_path, sep="\t", dtype=str, na_filter=False, chunksize=100_000
):
    chosen = chunk[chunk.source1_entity_id.isin(wanted_ids)]
    for source_id, ids_text in chosen.itertuples(index=False, name=None):
        truth[source_id] = set(filter(None, ids_text.split(",")))
missing_truth = wanted_ids - truth.keys()
if missing_truth:
    raise ValueError(f"Ground truth missing for {len(missing_truth)} sampled S1 rows")
del selected_s1, wanted_ids
gc.collect()
print(f"Training S1: {len(train_s1):,}; validation S1: {len(valid_s1):,}")
print("Training sample singletons:", sum(not truth[s] for s in train_s1.entity_id))


## 11. Build hard negative pair tables

Every validation candidate is scored. For training, the classifier sees all retrieved positives and the highest ranked retrieved negatives per S1.


In [ ]:
def make_pair_table(source_rows, retriever, truth_map, training=False):
    rows = []
    source_vectors = encode_source_frame(source_rows)
    semantic_scores, semantic_ids = retriever.embedding_search(source_vectors)
    if source_vectors is None:
        source_vectors = [None] * len(source_rows)
    true_total = 0
    true_retrieved = 0
    true_text_retrieved = 0
    true_embedding_retrieved = 0
    true_embedding_only = 0
    candidate_total = 0
    for i, (source, query_vector) in enumerate(
        zip(source_rows.itertuples(index=False), source_vectors), start=1
    ):
        source_id = source.entity_id
        positives = truth_map[source_id]
        prepared = prepare_record(source)
        hits = None if semantic_scores is None else (
            semantic_scores[i - 1], semantic_ids[i - 1]
        )
        candidates = retriever.retrieve(prepared, query_vector, hits)
        candidate_ids = {c["target_id"] for c in candidates}
        true_total += len(positives)
        true_retrieved += len(positives & candidate_ids)
        for candidate in candidates:
            if candidate["target_id"] in positives:
                text_hit = min(candidate["name_rank"], candidate["address_rank"],
                               candidate["prefix_rank"]) < 99
                embedding_hit = candidate["embedding_rank"] < 99
                true_text_retrieved += int(text_hit)
                true_embedding_retrieved += int(embedding_hit)
                true_embedding_only += int(embedding_hit and not text_hit)
        candidate_total += len(candidates)
        if training:
            positive_candidates = [c for c in candidates if c["target_id"] in positives]
            hard_negatives = [c for c in candidates if c["target_id"] not in positives]
            candidates = positive_candidates + hard_negatives[:NEGATIVES_PER_TRAIN_S1]
        for candidate in candidates:
            features = pair_features(prepared, candidate)
            features["source1_entity_id"] = source_id
            features["candidate_entity_id"] = candidate["target_id"]
            features["label"] = int(candidate["target_id"] in positives)
            rows.append(features)
        if i % 2_000 == 0:
            print(f"  {i:,}/{len(source_rows):,} S1 rows; {len(rows):,} pair rows")
    table = pd.DataFrame.from_records(rows)
    if table.empty:
        raise RuntimeError("Retrieval produced no candidate pairs")
    table[FEATURE_COLUMNS] = table[FEATURE_COLUMNS].astype("float32")
    diagnostics = {
        "queries": len(source_rows),
        "candidates_per_query": candidate_total / len(source_rows),
        "true_pairs": true_total,
        "retrieved_true_pairs": true_retrieved,
        "text_retrieved_true_pairs": true_text_retrieved,
        "embedding_retrieved_true_pairs": true_embedding_retrieved,
        "embedding_only_true_pairs": true_embedding_only,
        "blocking_recall": true_retrieved / true_total if true_total else 1.0,
    }
    return table, diagnostics

train_pairs, train_retrieval = make_pair_table(
    train_s1, train_retriever, truth, training=True
)
valid_pairs, valid_retrieval = make_pair_table(
    valid_s1, train_retriever, truth, training=False
)
train_retriever.close()
del train_retriever, train_embedding_index
gc.collect()
print("Train retrieval:", train_retrieval)
print("Validation retrieval:", valid_retrieval)
print("Train labels:", train_pairs.label.value_counts().to_dict())


## 12. Fit LightGBM and tune the macro F0.5 threshold

LightGBM is MIT licensed. Scores are evaluated per S1, including singleton cases. The validation threshold is selected after model fitting; it does not alter candidate generation.


In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=350,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.9,
    reg_lambda=2.0,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=-1,
)
model.fit(train_pairs[FEATURE_COLUMNS], train_pairs.label)
valid_pairs["score"] = model.predict_proba(valid_pairs[FEATURE_COLUMNS])[:, 1]
print("Model trained on", len(train_pairs), "retrieved pairs")
print("Validation positive pairs:", int(valid_pairs.label.sum()))


In [ ]:
validation_groups = {
    source_id: (
        group.candidate_entity_id.to_numpy(),
        group.score.to_numpy(dtype=np.float32),
    )
    for source_id, group in valid_pairs.groupby("source1_entity_id", sort=False)
}
valid_ids = valid_s1.entity_id.tolist()

def f05_for_sets(true_ids, predicted_ids):
    if not true_ids and not predicted_ids:
        return 1.0
    if not true_ids or not predicted_ids:
        return 0.0
    tp = len(true_ids & predicted_ids)
    return 1.25 * tp / (0.25 * len(true_ids) + len(predicted_ids))

def evaluate_threshold(threshold):
    scores = []
    tp = predicted_count = true_count = 0
    singleton_correct = singleton_total = 0
    for source_id in valid_ids:
        candidate_ids, probabilities = validation_groups.get(
            source_id, (np.empty(0, dtype=object), np.empty(0))
        )
        predicted = set(candidate_ids[probabilities >= threshold])
        actual = truth[source_id]
        scores.append(f05_for_sets(actual, predicted))
        tp += len(actual & predicted)
        predicted_count += len(predicted)
        true_count += len(actual)
        if not actual:
            singleton_total += 1
            singleton_correct += int(not predicted)
    return {
        "threshold": round(float(threshold), 3),
        "macro_f05": float(np.mean(scores)),
        "micro_precision": tp / predicted_count if predicted_count else 1.0,
        "micro_recall": tp / true_count if true_count else 1.0,
        "singleton_accuracy": singleton_correct / singleton_total if singleton_total else 1.0,
    }

sweep = pd.DataFrame(
    evaluate_threshold(t) for t in np.round(np.arange(0.10, 1.00, 0.01), 2)
)
best_row = sweep.sort_values(
    ["macro_f05", "threshold"], ascending=[False, False]
).iloc[0]
BEST_THRESHOLD = float(best_row.threshold)
print("Best validation result:")
display(best_row.to_frame().T)
print("Validation candidate recall:", round(valid_retrieval["blocking_recall"], 4))


## 13. Inspect validation mistakes

In [ ]:
def validation_errors(threshold, limit=20):
    results = []
    for source_id in valid_ids:
        candidate_ids, probabilities = validation_groups.get(
            source_id, (np.empty(0, dtype=object), np.empty(0))
        )
        predicted = set(candidate_ids[probabilities >= threshold])
        actual = truth[source_id]
        if predicted != actual:
            results.append({
                "source1_entity_id": source_id,
                "true_count": len(actual),
                "predicted_count": len(predicted),
                "false_positives": ",".join(sorted(predicted - actual)),
                "false_negatives": ",".join(sorted(actual - predicted)),
                "candidate_miss": int(bool(actual - set(candidate_ids))),
            })
    return pd.DataFrame(results).head(limit)

display(validation_errors(BEST_THRESHOLD))
display(pd.Series(model.feature_importances_, index=FEATURE_COLUMNS)
        .sort_values(ascending=False).head(15).to_frame("importance"))


## 14. Score every test Source 1 row and write the submission

The output loop holds one Source 1 batch and its pair features at a time. Every scored candidate is written to candidate_pairs.tsv, and every accepted match comes from that same list. An empty candidate list or no score above the threshold yields an empty match field.


In [ ]:
del train_pairs, valid_pairs, train_s1, valid_s1
gc.collect()

def write_test_outputs(retriever, test_source1, matching_path, candidate_path, threshold):
    total_rows = 0
    total_candidates = 0
    with matching_path.open("w", encoding="utf-8", newline="") as matching_file, \
         candidate_path.open("w", encoding="utf-8", newline="") as candidate_file:
        matching_writer = csv.writer(matching_file, delimiter="\t", lineterminator="\n")
        candidate_writer = csv.writer(candidate_file, delimiter="\t", lineterminator="\n")
        matching_writer.writerow(["source1_entity_id", "matched_entity_ids"])
        candidate_writer.writerow(["source1_entity_id", "candidate_entity_ids"])

        for start in range(0, len(test_source1), INFERENCE_BATCH_SIZE):
            batch = test_source1.iloc[start:start + INFERENCE_BATCH_SIZE]
            feature_rows = []
            batch_info = []
            query_vectors = encode_source_frame(batch)
            semantic_scores, semantic_ids = retriever.embedding_search(query_vectors)
            if query_vectors is None:
                query_vectors = [None] * len(batch)
            for row_index, (source, query_vector) in enumerate(
                zip(batch.itertuples(index=False), query_vectors)
            ):
                prepared = prepare_record(source)
                hits = None if semantic_scores is None else (
                    semantic_scores[row_index], semantic_ids[row_index]
                )
                candidates = retriever.retrieve(prepared, query_vector, hits)
                candidate_ids = [c["target_id"] for c in candidates]
                batch_info.append((source.entity_id, candidate_ids, len(feature_rows)))
                feature_rows.extend(pair_features(prepared, c) for c in candidates)
                total_candidates += len(candidates)

            if feature_rows:
                features = pd.DataFrame.from_records(
                    feature_rows, columns=FEATURE_COLUMNS
                ).astype("float32")
                probabilities = model.predict_proba(features)[:, 1]
            else:
                probabilities = np.empty(0, dtype=np.float32)

            for i, (source_id, candidate_ids, offset) in enumerate(batch_info):
                end = batch_info[i + 1][2] if i + 1 < len(batch_info) else len(probabilities)
                scores = probabilities[offset:end]
                matches = [
                    candidate_id for candidate_id, score in zip(candidate_ids, scores)
                    if score >= threshold
                ]
                candidate_writer.writerow([source_id, ",".join(candidate_ids)])
                matching_writer.writerow([source_id, ",".join(matches)])
                total_rows += 1
            if total_rows % 10_000 < len(batch):
                print(f"  Scored {total_rows:,}/{len(test_source1):,} test S1 rows")
    return total_rows, total_candidates

if RUN_TEST_INFERENCE:
    test_db_path = build_index("test")
    test_embedding_index = build_embedding_index(test_db_path, "test")
    test_retriever = CandidateRetriever(
        test_db_path, TOP_K_PER_CHANNEL, MAX_FINAL_CANDIDATES,
        test_embedding_index,
    )
    test_s1 = pd.read_csv(
        DATASET_DIR / "test/test_source1.tsv",
        sep="\t", dtype=str, na_filter=False, usecols=SOURCE_COLUMNS,
    )
    matching_path = OUTPUT_DIR / "matching_results.tsv"
    candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"
    try:
        n_rows, n_candidates = write_test_outputs(
            test_retriever, test_s1, matching_path, candidate_path, BEST_THRESHOLD
        )
    finally:
        test_retriever.close()
    print(f"Wrote {n_rows:,} Source 1 rows and {n_candidates:,} scored candidates")
    print(matching_path)
    print(candidate_path)


## 15. Validate the submission files

The supplied validator checks the full test IDs and the required match subset relation. It does not calculate F0.5.


In [ ]:
if RUN_TEST_INFERENCE:
    validator = DATASET_DIR.parent / "utils/validate_submission.py"
    if not validator.is_file():
        raise FileNotFoundError(f"Validator not found: {validator}")
    subprocess.run([
        sys.executable, str(validator),
        "--matching", str(matching_path),
        "--candidate", str(candidate_path),
        "--test-dir", str(DATASET_DIR / "test"),
    ], check=True)
